#  Atelier Scikit-learn 
Contexte 
Une entreprise possède plusieurs bâtiments équipés de capteurs IoT. Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la pression, la consommation énergétique, le bâtiment, la date et l'heure de la mesure.  Chaque mesure possède également un état (OK, ALERTE et ERREUR). L'objectif de l'atelier est de construire un modèle capable de prédire automatiquement l'état d'un capteur à partir de ses mesures. L'atelier suivra le workflow classique du Machine Learning : Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement → Modèle → fit()→ predict()→ Évaluation → Sauvegarde → Chargement → Réutilisation 

# Partie 0 – mise en place de l’environnement 
1) Structurer le projet comme suit : 

2) Stocker le fichier fourni mesures_capteurs.csv dans le sous-dossier data 
3) Créer le notebook atelier_scikit-learn_iot.ipynb 
4) Installer et importer seaborn, matplotlib et pandas 
5) Importer mesures_capteurs.csv dans le dataframe df 
6) Explorer le dataframe df 

In [27]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../data/mesures_capteurs.csv")
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    object 
 1   date_heure    605 non-null    object 
 2   id_capteur    605 non-null    object 
 3   batiment      605 non-null    object 
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    object 
dtypes: float64(4), object(5)
memory usage: 42.7+ KB


In [29]:
df.shape

(605, 9)

# Partie 1 – Gestion des doublons Avec Pandas, 
1) vérifier l’existence de doublons dans df 
2) le cas échéant, supprimer les doublons puis vérifier la suppression  

## 1) vérifier l’existence de doublons dans df 


In [30]:
print("Nombre de doublons :", df.duplicated().sum())


Nombre de doublons : 5


## 2) le cas échéant, supprimer les doublons puis vérifier la suppression  

In [31]:
df = df.drop_duplicates()
print("Nouvelles dimensions :", df.shape)

Nouvelles dimensions : (600, 9)


# Partie 2 – Sélection de y (cible) et X (caractéristiques) 
1) Définir "etat" comme la cible ou valeur à prédire et "temperature", "humidite", "pression" et "consommation" comme caractéristiques ou variables explicatives 
2) Afficher les cinq premières lignes de X et de y 
3) Quel est le type du problème de machine learning ?  

## 1) Définir "etat" comme la cible ou valeur à prédire et "temperature", "humidite", "pression" et "consommation" comme caractéristiques ou variables explicatives 

In [40]:
# definir X Y 
X = df[["temperature", "humidite", "pression", "consommation"]]
y = df["etat"]

## 2) Afficher les cinq premières lignes de X et de y 


In [33]:
print("les features")
print(X.head())
print("la cible")
print(y.head())

les features
   temperature  humidite  pression  consommation
0        25.46     58.06   1008.95        287.28
1        24.00     79.73    993.39        116.20
2        25.82     54.47   1010.32        288.50
3        28.23     69.39   1019.62        136.65
4        20.58     53.80   1016.58        182.62
la cible
0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: object


In [34]:
# les elements de atat 
y.unique() 

array(['OK', nan, 'ALERTE', 'ERREUR'], dtype=object)

## 3) Quel est le type du problème de machine learning ?  
Il s'agit d'un problème de classification multi-classes : la cible "etat" contient 3 
catégories possibles (OK, ALERTE, ERREUR), pas un nombre continu à prédire.

# Partie 3 – Découpage Train/Test Diviser X en deux ensembles distincts :
 un pour l'entraînement (train) et un pour le test (test). Avec les conditions suivantes : 20% des données serviront au test ; garantir la reproductibilité du découpage ; conserver les mêmes proportions de classes dans l'ensemble de train et de test que dans les données d'origine. 

In [35]:
y.isnull().sum()

4

In [36]:
# Vérifie AVANT de supprimer si les NaN sont concentrés sur un bâtiment, une période...
lignes_nan = df[df["etat"].isnull()]
print(lignes_nan["batiment"].value_counts())

batiment
B004    2
B003    1
B002    1
Name: count, dtype: int64


In [39]:
y.unique()

array(['OK', nan, 'ALERTE', 'ERREUR'], dtype=object)

In [44]:
# supprime UNIQUEMENT les lignes où "etat" (la cible) manque
df = df.dropna(subset=["etat"])

In [45]:
y.unique()

array(['OK', 'ALERTE', 'ERREUR'], dtype=object)

In [47]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,        # 20% des données pour le test
    random_state=42,      # garantir la reproductibilité
    stratify=y            # conserver les mêmes proportions de classes
)

print("Taille X_train :", X_train.shape)
print("Taille X_test  :", X_test.shape)

Taille X_train : (476, 4)
Taille X_test  : (120, 4)


In [48]:
y_train.unique()

array(['OK', 'ALERTE', 'ERREUR'], dtype=object)

In [49]:
y_test.unique()

array(['OK', 'ALERTE', 'ERREUR'], dtype=object)

# Partie 4 – Gestion des valeurs manquantes 
1) Vérifier l’existence de valeurs manquantes 
2) Sélectionner SimpleImputer avec la médiane 
3) Qu’est ce qui justifie le choix de la médiane ? 
4) Trouver les paramètres (médianes) de l’imputeur sur X_train 
5) Déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test 

## 1) Vérifier l’existence de valeurs manquantes 


In [53]:
print("Valeurs manquantes dans X_train :")
print(X_train.isnull().sum())
print("\nValeurs manquantes dans X_test :")
print(X_test.isnull().sum())

Valeurs manquantes dans X_train :
temperature     5
humidite        4
pression        5
consommation    3
dtype: int64

Valeurs manquantes dans X_test :
temperature     1
humidite        1
pression        0
consommation    2
dtype: int64


## 2) Sélectionner SimpleImputer avec la médiane 


In [ ]:
# Création de l'imputeur, configuré pour remplacer par la MEDIANE

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

## 3) Qu’est ce qui justifie le choix de la médiane ? 

La médiane est moins sensible aux valeurs extrêmes que la moyenne. Rappelle-toi les 
anomalies déjà identifiées dans ce dataset (température ~-18°C ou ~58°C, liées à 
B004) : si on utilisait la moyenne pour imputer, ces valeurs aberrantes tireraient 
la moyenne dans leur direction, faussant les valeurs de remplacement. La médiane 
reste stable face à ces cas isolés.

## 4) Trouver les paramètres (médianes) de l’imputeur sur X_train 


In [55]:
# fit() calcule les médianes UNIQUEMENT sur X_train, jamais sur X_test,
# pour éviter toute fuite de données (data leakage) vers le jeu de test
imputer.fit(X_train)
print("Médianes calculées sur X_train :", imputer.statistics_)

Médianes calculées sur X_train : [  24.9    65.38 1012.3   206.59]


## 5) Déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test 

In [56]:
# transform() applique ensuite ces mêmes médianes à train ET à test
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

In [57]:
print("Valeurs manquantes restantes (train) :", pd.DataFrame(X_train_imputed).isnull().sum().sum())
print("Valeurs manquantes restantes (test)  :", pd.DataFrame(X_test_imputed).isnull().sum().sum())

Valeurs manquantes restantes (train) : 0
Valeurs manquantes restantes (test)  : 0


# Partie 5 – Mise à l'échelle 
1) Sélectionner StandardScaler pour mettre à l’échelle les transformés de l’imputation 
2) Qu’est ce qui justifie la standardisation ? 
3) Trouver les paramètres (moyennes et écart-types) du scaleur sur X_train_imputed 
4) Déterminer X_train_scaled et X_test_scaled, les transformés de X_train_imputed et X_test_imputed 

## 1) Sélectionner StandardScaler pour mettre à l’échelle les transformés de l’imputation 


In [58]:
from sklearn.preprocessing import StandardScaler

# Création du scaler, configuré pour centrer-réduire (standardisation)
scaler = StandardScaler()

## 2) Qu’est ce qui justifie la standardisation ? 

La standardisation est moins sensible aux valeurs extrêmes que la normalisation 
Min-Max. Rappelle-toi les anomalies déjà identifiées (température ~-18°C ou ~58°C) : 
avec une normalisation Min-Max, ces valeurs extrêmes deviendraient le nouveau min/max, 
écrasant l'échelle de toutes les autres mesures normales vers un intervalle très 
restreint. La standardisation reste plus stable face à ce type de cas.


## 3) Trouver les paramètres (moyennes et écart-types) du scaleur sur X_train_imputed 


In [59]:
# fit() calcule moyenne et écart-type UNIQUEMENT sur X_train_imputed,
# pour la même raison qu'avec l'imputer : éviter la fuite de données vers le test
scaler.fit(X_train_imputed)

print("Moyennes calculées sur X_train_imputed :", scaler.mean_)
print("Écarts-types calculés :", scaler.scale_)

Moyennes calculées sur X_train_imputed : [  24.9637395    64.85044118 1012.07621849  210.15394958]
Écarts-types calculés : [ 4.19303979 10.16279173 10.9941836  73.42092322]


## 4) Déterminer X_train_scaled et X_test_scaled, les transformés de X_train_imputed et X_test_imputed 

In [60]:
# transform() applique ensuite ces mêmes paramètres à train ET à test
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Vérification : après standardisation, la moyenne doit être ~0 et l'écart-type ~1
print("Moyenne après scaling (train) :", X_train_scaled.mean(axis=0).round(2))
print("Écart-type après scaling (train) :", X_train_scaled.std(axis=0).round(2))

Moyenne après scaling (train) : [-0.  0. -0.  0.]
Écart-type après scaling (train) : [1. 1. 1. 1.]
